# 02 · Data Understanding (CRISP-DM)

**Project:** Pemetaan Keluhan Infrastruktur & Prioritas Perbaikan (AOL Business Intelligence, Group 9)
**Topik kelompok:** keluhan jalan rusak di DKI Jakarta (daratan), sumber X/Twitter 2019 – sekarang

Notebook ini mengikuti empat tugas fase *Data Understanding* di CRISP-DM:

| No | Tugas CRISP-DM | Bagian notebook |
|---|---|---|
| 1 | *Collect initial data* | Sumber data & alur pembersihan |
| 2 | *Describe data* | Struktur dataset & data dictionary |
| 3 | *Explore data* | Waktu, wilayah, urgensi, teks |
| 4 | *Verify data quality* | Kelengkapan, lokasi, label, keseimbangan kelas |

**Input:** `data/processed/jalan_rusak/` (hasil `src/preprocessing/preprocess_jalan_rusak.py`) dan `data/external/batas-wilayah/`.
**Output:** grafik di `reports/figures/du_*.png` dan tabel di `reports/tables/du_*.csv`, siap ditempel ke laporan.

> Notebook ini sengaja **tidak menampilkan isi tweet atau username**. Jangan menambah sel yang mencetak `full_text`/`akun_media` sebelum commit.

In [ ]:
import json
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch, Polygon

# Cari root repo dari lokasi notebook (bisa dijalankan dari notebooks/ atau root)
ROOT = Path.cwd()
while not ((ROOT / "README.md").exists() and (ROOT / "data").is_dir()):
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Root repo tidak ketemu. Jalankan notebook dari dalam folder repo.")
    ROOT = ROOT.parent

PROC = ROOT / "data" / "processed" / "jalan_rusak"
BATAS = ROOT / "data" / "external" / "batas-wilayah"
FIG = ROOT / "reports" / "figures"
TAB = ROOT / "reports" / "tables"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)
print("Root repo:", ROOT)

In [ ]:
# ---------- Gaya grafik (satu sistem warna untuk semua grafik laporan) ----------
SURFACE, INK, INK2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
BIRU, ORANYE, HIJAU, KUNING, MAGENTA = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"
SEQ = ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]  # sekuensial biru, terang -> gelap
NOL = "#f0efec"                                                   # area bernilai 0 / tanpa data
URG_WARNA = {"Tinggi": "#104281", "Sedang": "#3987e5", "Rendah": "#9ec5f4", "Belum berlabel": "#c3c2b7"}
AKUN_WARNA = {"warga": BIRU, "media_resmi": ORANYE}
AKUN_LABEL = {"warga": "Warga", "media_resmi": "Media / akun resmi"}

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "legend.fontsize": 9,
})


def angka(v, d=0):
    # Format angka gaya Indonesia: 3.935 dan 7,9
    s = f"{v:,.{d}f}"
    return s.replace(",", "_").replace(".", ",").replace("_", ".")


def judul(ax, teks, sub=None):
    ax.set_title(teks, pad=20 if sub else 8)
    if sub:
        ax.text(0, 1.02, sub, transform=ax.transAxes, fontsize=9, color=INK2)


def simpan(fig, nama):
    path = FIG / f"{nama}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print("Tersimpan:", path.relative_to(ROOT))


def label_bar(ax, bars, d=0, horizontal=False):
    for b in bars:
        v = b.get_width() if horizontal else b.get_height()
        if v == 0:
            continue
        if horizontal:
            ax.text(v, b.get_y() + b.get_height() / 2, " " + angka(v, d), va="center", fontsize=8, color=INK2)
        else:
            ax.text(b.get_x() + b.get_width() / 2, v, angka(v, d), ha="center", va="bottom", fontsize=8, color=INK2)

In [ ]:
# ---------- Load data ----------
dtype = {"id_str": str, "adm4_pcode": str, "user_hash": str}
keluhan = pd.read_csv(PROC / "keluhan_jalan_rusak.csv", dtype=dtype)
kandidat = pd.read_csv(PROC / "semua_kandidat.csv", dtype=dtype)
perbaikan = pd.read_csv(PROC / "laporan_perbaikan.csv", dtype=dtype)
agg_kel = pd.read_csv(PROC / "agregat_kelurahan.csv", dtype={"adm4_pcode": str})
agg_kec = pd.read_csv(PROC / "agregat_kecamatan.csv")

for nama, d in [("keluhan_jalan_rusak", keluhan), ("semua_kandidat", kandidat),
                ("laporan_perbaikan", perbaikan), ("agregat_kelurahan", agg_kel), ("agregat_kecamatan", agg_kec)]:
    print(f"{nama:22} {d.shape[0]:>6,} baris x {d.shape[1]:>2} kolom")

## 1. Collect Initial Data

Data dikumpulkan dari X (Twitter) dengan **tweet-harvest** dalam dua tahap:

1. **Scraping awal** per anggota (personal assignment): jalan rusak (Fio), fasilitas publik (Mail), kemacetan (Angel). Dari data Mail dan Angel hanya tweet yang menyebut jalan rusak yang diambil.
2. **Scraping ulang per kecamatan**: 10 frasa kerusakan jalan × 42 kecamatan DKI (`src/scraping/generate_queries.py`), supaya sebaran wilayah tidak bergantung pada satu query umum.

Data pendukung: batas wilayah kelurahan BPS (via HDX) dan estimasi penduduk 2025 per kelurahan (lihat `data/external/README.md`).

In [ ]:
# Alur pembersihan diambil dari ringkasan.md yang ditulis pipeline (tahap sebelum semua_kandidat tidak tersimpan per baris)
teks_ringkasan = (PROC / "ringkasan.md").read_text(encoding="utf-8")
blok = teks_ringkasan.split("## Alur pembersihan")[1].split("##")[0]
funnel = [tuple(c.strip() for c in baris.strip("|").split("|"))
          for baris in blok.strip().splitlines()[2:] if baris.startswith("|")]
funnel = pd.DataFrame(funnel, columns=["tahap", "jumlah"]).astype({"jumlah": int})
funnel["sisa_%"] = (funnel["jumlah"] / funnel["jumlah"].iloc[0] * 100).round(1)
funnel.to_csv(TAB / "du_alur_pembersihan.csv", index=False)

blok_sumber = teks_ringkasan.split("## Sumber data")[1].split("##")[0]
sumber = [tuple(c.strip() for c in b.strip("|").split("|")) for b in blok_sumber.strip().splitlines()[2:] if b.startswith("|")]
sumber = pd.DataFrame(sumber, columns=["sumber", "baris_mentah", "dipakai_setelah_filter_keyword"])
sumber["keluhan_final"] = sumber["sumber"].map(keluhan["sumber"].value_counts()).fillna(0).astype(int)
sumber.to_csv(TAB / "du_sumber_data.csv", index=False)
display(sumber)
display(funnel)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))
y = np.arange(len(funnel))[::-1]
bars = ax.barh(y, funnel["jumlah"], color=[BIRU] * (len(funnel) - 1) + ["#104281"], height=0.62)
ax.set_yticks(y, funnel["tahap"])
ax.tick_params(axis="y", colors=INK2, length=0)
ax.grid(axis="y", visible=False)
for b, pct in zip(bars, funnel["sisa_%"]):
    ax.text(b.get_width(), b.get_y() + b.get_height() / 2, f"  {angka(b.get_width())}  ({pct:.0f}%)",
            va="center", fontsize=8, color=INK2)
ax.set_xlim(0, funnel["jumlah"].max() * 1.22)
ax.set_xlabel("Jumlah tweet")
judul(ax, "Alur pembersihan data",
      f"Dari {angka(funnel['jumlah'].iloc[0])} tweet mentah tersisa {angka(funnel['jumlah'].iloc[-1])} keluhan jalan rusak")
simpan(fig, "du_01_alur_pembersihan")
plt.show()

In [ ]:
# Alasan baris dibuang (dicatat per baris di semua_kandidat.csv)
alasan = kandidat["alasan_dibuang"].fillna("(lolos)").value_counts().rename_axis("alasan").reset_index(name="jumlah")
lolos = kandidat[kandidat["alasan_dibuang"].isna()]
print("Dari baris yang lolos filter, jenis konten:")
display(lolos["jenis"].value_counts())
display(alasan)

## 2. Describe Data

Unit analisis: **satu baris = satu tweet keluhan jalan rusak** di `keluhan_jalan_rusak.csv`. Tabel agregat menyediakan unit spasial **kelurahan (261)** dan **kecamatan (42)**, termasuk wilayah tanpa keluhan, lengkap dengan estimasi penduduk 2025.

In [ ]:
DESKRIPSI = {
    "id_str": "ID unik tweet (string 19 digit)",
    "created_at": "Waktu tweet (WIB)",
    "tahun": "Tahun tweet", "bulan": "Bulan tweet (1-12)", "periode": "Tahun-bulan (YYYY-MM)",
    "jam": "Jam tweet (0-23, WIB)", "hari": "Nama hari (bahasa Inggris)", "akhir_pekan": "True jika Sabtu/Minggu",
    "sumber": "Asal data: fio_awal, scrape_kecamatan, mail_fasilitas",
    "tipe_akun": "warga atau media_resmi (media, instansi, akun info)",
    "akun_media": "Username bila akun media/resmi; kosong untuk warga",
    "user_hash": "ID pengguna yang di-hash (anonim)",
    "jenis": "Jenis konten; di file ini selalu 'keluhan'",
    "full_text": "Teks asli tweet",
    "text_clean": "Teks setelah dibersihkan (lowercase, URL/mention dibuang, slang dinormalisasi)",
    "text_model": "text_clean setelah stopword removal + stemming (Sastrawi), input fitur teks model",
    "panjang_teks": "Jumlah kata di text_clean",
    "kota": "Kota administrasi hasil ekstraksi lokasi", "kecamatan": "Kecamatan hasil ekstraksi lokasi",
    "kelurahan": "Kelurahan hasil ekstraksi lokasi", "adm4_pcode": "Kode BPS kelurahan, kunci join ke peta",
    "level_lokasi": "Lokasi terdalam yang ditemukan: kelurahan/kecamatan/kota/tidak_ada",
    "sumber_lokasi": "Cara lokasi ditemukan: gazetteer, kawasan, nama_kota",
    "ambigu_kelurahan": "True jika kecamatan pasti tapi kelurahan hanya tebakan",
    "lokasi_ambigu": "True jika tweet menyebut lebih dari satu kecamatan/kota",
    "nama_jalan": "Nama jalan yang disebut (regex)",
    "status_wilayah": "dki (lokasi DKI ditemukan) atau jakarta_umum (hanya menyebut Jakarta)",
    "flag_luar_jakarta": "True jika menyebut wilayah luar Jakarta",
    "urgensi_manual": "Label urgensi anotasi manual (Tinggi/Sedang/Rendah)",
    "urgensi_aturan": "Label urgensi dari aturan kata penanda",
    "urgensi": "Label akhir: manual jika ada, jika tidak dari aturan",
    "sumber_label": "manual / aturan / kosong (belum berlabel)",
    "favorite_count": "Jumlah like", "retweet_count": "Jumlah retweet", "reply_count": "Jumlah balasan",
    "quote_count": "Jumlah quote", "engagement": "Total like + retweet + balasan + quote",
    "ada_gambar": "True jika tweet memuat gambar",
}
TEKS_SENSITIF = {"full_text", "text_clean", "text_model", "akun_media", "user_hash", "id_str", "nama_jalan"}


def contoh(s, kol):
    if kol in TEKS_SENSITIF:
        return "(teks, tidak ditampilkan)"
    v = s.dropna().unique()[:3]
    return ", ".join(map(str, v))


kamus = pd.DataFrame({
    "kolom": keluhan.columns,
    "tipe": [str(t) for t in keluhan.dtypes],
    "terisi_%": (keluhan.notna().mean() * 100).round(1).values,
    "nilai_unik": keluhan.nunique().values,
    "contoh": [contoh(keluhan[c], c) for c in keluhan.columns],
    "deskripsi": [DESKRIPSI.get(c, "") for c in keluhan.columns],
})
kamus.to_csv(TAB / "du_data_dictionary.csv", index=False)
pd.set_option("display.max_colwidth", 80)
display(kamus)

In [ ]:
numerik = ["panjang_teks", "favorite_count", "retweet_count", "reply_count", "quote_count", "engagement"]
statistik = keluhan[numerik].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).T.round(1)
statistik.to_csv(TAB / "du_statistik_numerik.csv")
display(statistik)

## 3. Explore Data

### 3.1 Pola waktu

In [ ]:
per_tahun = keluhan.pivot_table(index="tahun", columns="tipe_akun", values="id_str", aggfunc="count", fill_value=0)
per_tahun = per_tahun[["warga", "media_resmi"]]

fig, ax = plt.subplots(figsize=(9, 4.2))
bawah = np.zeros(len(per_tahun))
for akun in per_tahun.columns:
    ax.bar(per_tahun.index, per_tahun[akun], bottom=bawah, width=0.62, color=AKUN_WARNA[akun],
           label=AKUN_LABEL[akun], edgecolor=SURFACE, linewidth=2)
    bawah += per_tahun[akun].values
for x, total in zip(per_tahun.index, bawah):
    ax.text(x, total, f"{int(total)}", ha="center", va="bottom", fontsize=8, color=INK2)
ax.set_xticks(per_tahun.index)
ax.grid(axis="x", visible=False)
ax.set_ylabel("Jumlah keluhan")
ax.legend(loc="upper left", ncols=2)
judul(ax, "Keluhan jalan rusak per tahun", "Dipisah menurut tipe akun pengirim")
simpan(fig, "du_02_keluhan_per_tahun")
plt.show()
display(per_tahun.assign(total=per_tahun.sum(axis=1)))

In [ ]:
URUT_HARI = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
NAMA_HARI = ["Sen", "Sel", "Rab", "Kam", "Jum", "Sab", "Min"]
per_hari = keluhan["hari"].value_counts().reindex(URUT_HARI, fill_value=0)
per_jam = keluhan["jam"].value_counts().reindex(range(24), fill_value=0)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1, 2.4]})
label_bar(a1, a1.bar(NAMA_HARI, per_hari.values, color=BIRU, width=0.62))
a1.grid(axis="x", visible=False)
a1.set_ylabel("Jumlah keluhan")
judul(a1, "Per hari")
a2.bar(per_jam.index, per_jam.values, color=BIRU, width=0.7)
a2.set_xticks(range(0, 24, 2))
a2.set_xlabel("Jam (WIB)")
a2.grid(axis="x", visible=False)
judul(a2, "Per jam")
fig.suptitle("Kapan keluhan diunggah", x=0.01, ha="left", fontweight="bold", fontsize=12)
fig.tight_layout()
simpan(fig, "du_03_pola_hari_jam")
plt.show()
print(f"Akhir pekan: {keluhan['akhir_pekan'].mean():.1%} keluhan")

### 3.2 Sebaran wilayah

In [ ]:
pend_kota = agg_kel.groupby("kota")["penduduk_2025_est"].sum()
per_kota = keluhan["kota"].value_counts().reindex(pend_kota.index, fill_value=0)
tabel_kota = pd.DataFrame({"keluhan": per_kota, "penduduk_2025_est": pend_kota})
tabel_kota["per_100rb"] = (tabel_kota["keluhan"] / tabel_kota["penduduk_2025_est"] * 1e5).round(2)
tabel_kota = tabel_kota.sort_values("per_100rb", ascending=False)
tanpa_kota = int(keluhan["kota"].isna().sum())

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
nama = [k.replace("Jakarta ", "Jak. ") for k in tabel_kota.index]
y = np.arange(len(tabel_kota))[::-1]
label_bar(a1, a1.barh(y, tabel_kota["keluhan"], color=BIRU, height=0.6), horizontal=True)
a1.barh(-1.2, tanpa_kota, color="#c3c2b7", height=0.6)
a1.text(tanpa_kota, -1.2, f" {tanpa_kota}", va="center", fontsize=8, color=INK2)
a1.set_yticks(list(y) + [-1.2], nama + ["(tanpa lokasi)"])
a1.set_xlim(0, max(tabel_kota["keluhan"].max(), tanpa_kota) * 1.18)
judul(a1, "Jumlah keluhan")
label_bar(a2, a2.barh(y, tabel_kota["per_100rb"], color="#104281", height=0.6), d=1, horizontal=True)
a2.set_yticks(y, nama)
a2.set_xlim(0, tabel_kota["per_100rb"].max() * 1.2)
judul(a2, "Per 100 ribu penduduk")
for a in (a1, a2):
    a.grid(axis="y", visible=False)
    a.tick_params(axis="y", colors=INK2, length=0)
fig.suptitle("Keluhan per kota administrasi", x=0.01, ha="left", fontweight="bold", fontsize=12)
fig.tight_layout()
simpan(fig, "du_04_keluhan_per_kota")
plt.show()
display(tabel_kota)

In [ ]:
# ---------- Peta choropleth (tanpa geopandas, cukup matplotlib) ----------
def baca_geojson(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))["features"]


def poligon(geom):
    return [geom["coordinates"]] if geom["type"] == "Polygon" else geom["coordinates"]


def gambar_peta(ax, fitur, warna_fn, garis_luar=None, lw=0.4):
    for f in fitur:
        for p in poligon(f["geometry"]):
            ax.add_patch(Polygon(p[0], closed=True, fc=warna_fn(f["properties"]), ec=SURFACE, lw=lw))
    for f in garis_luar or []:
        for p in poligon(f["geometry"]):
            ax.add_patch(Polygon(p[0], closed=True, fill=False, ec=INK, lw=0.9))
    ax.autoscale()
    ax.set_aspect("equal")
    ax.axis("off")


def kelas(nilai, batas):
    # 0 -> -1 (abu), lalu indeks kelas sesuai batas
    return -1 if nilai <= 0 else int(np.searchsorted(batas, nilai, side="right"))


geo_kota = baca_geojson(BATAS / "jakarta_adm2.geojson")
geo_kec = baca_geojson(BATAS / "jakarta_adm3.geojson")
geo_kel = baca_geojson(BATAS / "jakarta_adm4.geojson")
print(len(geo_kota), "kota |", len(geo_kec), "kecamatan |", len(geo_kel), "kelurahan")

In [ ]:
# Kecamatan: keluhan per 100 ribu penduduk
kec_rate = agg_kec.set_index("kecamatan")["keluhan_per_100rb"]
BATAS_KEC = [1, 2, 4, 6]
LBL_KEC = ["< 1", "1 – 2", "2 – 4", "4 – 6", "≥ 6"]


def warna_kec(p):
    k = kelas(kec_rate.get(p["adm3_name"], 0), BATAS_KEC)
    return NOL if k < 0 else SEQ[k]


tidak_cocok = sorted(set(f["properties"]["adm3_name"] for f in geo_kec) - set(kec_rate.index))
print("Kecamatan di peta tapi tidak ada di agregat:", tidak_cocok or "-")

fig, ax = plt.subplots(figsize=(8, 8))
gambar_peta(ax, geo_kec, warna_kec, garis_luar=geo_kota, lw=0.6)
hit = Counter(kelas(kec_rate.get(f["properties"]["adm3_name"], 0), BATAS_KEC) for f in geo_kec)
leg = [Patch(fc=NOL, ec="#c3c2b7", label=f"0  ({hit[-1]})")] + \
      [Patch(fc=SEQ[i], label=f"{LBL_KEC[i]}  ({hit[i]})") for i in range(len(SEQ))]
ax.legend(handles=leg, loc="lower left", title="keluhan / 100 rb penduduk\n(jumlah kecamatan)", title_fontsize=9,
          alignment="left", labelcolor=INK2)
judul(ax, "Keluhan jalan rusak per 100 ribu penduduk, per kecamatan",
      "Hanya keluhan yang lokasinya ditemukan sampai kecamatan/kelurahan. Penduduk: estimasi 2025")
simpan(fig, "du_05_peta_kecamatan_per100rb")
plt.show()

In [ ]:
# Kelurahan: jumlah keluhan (menunjukkan seberapa jarang data di level kelurahan)
kel_n = agg_kel.set_index("adm4_pcode")["jumlah_keluhan"]
BATAS_KEL = [2, 4, 8, 16]
LBL_KEL = ["1", "2 – 3", "4 – 7", "8 – 15", "≥ 16"]


def warna_kel(p):
    k = kelas(kel_n.get(p["adm4_pcode"], 0), BATAS_KEL)
    return NOL if k < 0 else SEQ[k]


fig, ax = plt.subplots(figsize=(8, 8))
gambar_peta(ax, geo_kel, warna_kel, garis_luar=geo_kota, lw=0.35)
hit = Counter(kelas(kel_n.get(f["properties"]["adm4_pcode"], 0), BATAS_KEL) for f in geo_kel)
leg = [Patch(fc=NOL, ec="#c3c2b7", label=f"0  ({hit[-1]})")] + \
      [Patch(fc=SEQ[i], label=f"{LBL_KEL[i]}  ({hit[i]})") for i in range(len(SEQ))]
ax.legend(handles=leg, loc="lower left", title="jumlah keluhan\n(jumlah kelurahan)", title_fontsize=9,
          alignment="left", labelcolor=INK2)
judul(ax, "Jumlah keluhan jalan rusak per kelurahan",
      f"{(kel_n > 0).sum()} dari {len(kel_n)} kelurahan punya minimal 1 keluhan")
simpan(fig, "du_06_peta_kelurahan_jumlah")
plt.show()

top_kel = agg_kel.sort_values("jumlah_keluhan", ascending=False).head(10)[
    ["kelurahan", "kecamatan", "kota", "jumlah_keluhan", "keluhan_per_100rb", "laporan_perbaikan"]]
display(top_kel)

### 3.3 Label urgensi

In [ ]:
URUT_URG = ["Tinggi", "Sedang", "Rendah"]
lab = keluhan.assign(urgensi=keluhan["urgensi"].fillna("Belum berlabel"),
                     sumber_label=keluhan["sumber_label"].fillna("belum"))
tab_urg = pd.crosstab(lab["urgensi"], lab["sumber_label"]).reindex(URUT_URG + ["Belum berlabel"], fill_value=0)
tab_urg = tab_urg.reindex(columns=["manual", "aturan", "belum"], fill_value=0)

fig, ax = plt.subplots(figsize=(9, 3.8))
y = np.arange(len(tab_urg))[::-1]
kiri = np.zeros(len(tab_urg))
for kol, warna, nama in [("manual", "#104281", "Label manual"), ("aturan", "#86b6ef", "Label aturan"),
                         ("belum", "#c3c2b7", "Belum berlabel")]:
    ax.barh(y, tab_urg[kol], left=kiri, color=warna, height=0.6, label=nama, edgecolor=SURFACE, linewidth=2)
    kiri += tab_urg[kol].values
for yy, total in zip(y, kiri):
    ax.text(total, yy, f"  {int(total)}  ({total / len(lab):.0%})", va="center", fontsize=8, color=INK2)
ax.set_yticks(y, tab_urg.index)
ax.tick_params(axis="y", colors=INK2, length=0)
ax.grid(axis="y", visible=False)
ax.set_xlim(0, kiri.max() * 1.25)
ax.legend(loc="lower right")
judul(ax, "Distribusi label urgensi", "Label manual menimpa label aturan bila keduanya ada")
simpan(fig, "du_07_distribusi_urgensi")
plt.show()
display(tab_urg.assign(total=tab_urg.sum(axis=1)))
display(pd.crosstab(lab["tipe_akun"], lab["urgensi"]).reindex(columns=URUT_URG + ["Belum berlabel"]))

In [ ]:
# Kata pembeda tiap kelas urgensi (dari text_model = teks yang sudah di-stem)
# Skor = proporsi dokumen di kelas itu yang memuat kata - proporsi di kelas lain. Kata muncul < 5 dokumen diabaikan.
UMUM = {"jalan", "rusak", "jl", "jakarta", "lubang", "aspal"}
berlabel = keluhan.dropna(subset=["urgensi"]).copy()
berlabel["kata"] = berlabel["text_model"].fillna("").str.split().map(lambda t: set(t) - UMUM)
df_kata = Counter(w for s in berlabel["kata"] for w in s)

fig, axes = plt.subplots(1, 3, figsize=(12, 4.2), sharex=True)
for ax, kls in zip(axes, URUT_URG):
    di, luar = berlabel[berlabel["urgensi"] == kls], berlabel[berlabel["urgensi"] != kls]
    skor = {}
    for w, n in df_kata.items():
        if n < 5:
            continue
        skor[w] = di["kata"].map(lambda s: w in s).mean() - luar["kata"].map(lambda s: w in s).mean()
    top = pd.Series(skor).sort_values(ascending=False).head(10)[::-1]
    ax.barh(top.index, top.values * 100, color=URG_WARNA[kls], height=0.6)
    ax.tick_params(axis="y", colors=INK2, length=0)
    ax.grid(axis="y", visible=False)
    judul(ax, f"{kls}  (n = {len(di)})")
axes[1].set_xlabel("Selisih % dokumen yang memuat kata (kelas ini vs kelas lain)")
fig.suptitle("Kata paling membedakan tiap kelas urgensi", x=0.01, ha="left", fontweight="bold", fontsize=12)
fig.tight_layout()
simpan(fig, "du_08_kata_pembeda_urgensi")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
data_box = [keluhan.loc[keluhan["urgensi"] == k, "panjang_teks"] for k in URUT_URG]
bp = ax.boxplot(data_box, vert=False, patch_artist=True, widths=0.5, medianprops={"color": INK},
                flierprops={"markersize": 3, "markerfacecolor": MUTED, "markeredgecolor": MUTED})
for patch, k in zip(bp["boxes"], URUT_URG):
    patch.set_facecolor(URG_WARNA[k])
    patch.set_edgecolor(INK2)
ax.set_yticks([1, 2, 3], URUT_URG)
ax.grid(axis="y", visible=False)
ax.set_xlabel("Jumlah kata (text_clean)")
judul(ax, "Panjang teks keluhan per kelas urgensi")
simpan(fig, "du_09_panjang_teks_urgensi")
plt.show()
display(keluhan.groupby("urgensi")["panjang_teks"].describe().reindex(URUT_URG).round(1))

## 4. Verify Data Quality

In [ ]:
# 4.1 Kelengkapan kolom & duplikasi
hilang = keluhan.isna().sum().rename("kosong").to_frame()
hilang["kosong_%"] = (hilang["kosong"] / len(keluhan) * 100).round(1)
hilang = hilang[hilang["kosong"] > 0].sort_values("kosong", ascending=False)
hilang.to_csv(TAB / "du_kolom_kosong.csv")
display(hilang)
print("Duplikat id_str          :", keluhan["id_str"].duplicated().sum())
print("Duplikat text_clean      :", keluhan["text_clean"].duplicated().sum())

In [ ]:
# 4.2 Kualitas lokasi
URUT_LEVEL = ["kelurahan", "kecamatan", "kota", "tidak_ada"]
level = keluhan["level_lokasi"].value_counts().reindex(URUT_LEVEL, fill_value=0)
amb_kel = int(keluhan["ambigu_kelurahan"].astype(bool).sum())
amb_lok = int(keluhan["lokasi_ambigu"].astype(bool).sum())

fig, ax = plt.subplots(figsize=(8, 3.4))
warna = [SEQ[4], SEQ[3], SEQ[1], "#c3c2b7"]
y = np.arange(len(level))[::-1]
bars = ax.barh(y, level.values, color=warna, height=0.6)
for b in bars:
    ax.text(b.get_width(), b.get_y() + b.get_height() / 2,
            f"  {int(b.get_width())}  ({b.get_width() / len(keluhan):.0%})", va="center", fontsize=8, color=INK2)
ax.set_yticks(y, ["Sampai kelurahan", "Sampai kecamatan", "Hanya kota", "Tanpa lokasi"])
ax.tick_params(axis="y", colors=INK2, length=0)
ax.grid(axis="y", visible=False)
ax.set_xlim(0, level.max() * 1.25)
judul(ax, "Ketelitian lokasi keluhan",
      f"Dari yang berlokasi: {amb_kel} keluhan kelurahannya tebakan, {amb_lok} menyebut lebih dari satu wilayah")
simpan(fig, "du_10_ketelitian_lokasi")
plt.show()
display(keluhan["sumber_lokasi"].fillna("tidak_ada").value_counts())

In [ ]:
# 4.3 Konsistensi label: aturan vs manual (hanya baris yang punya keduanya)
dua = keluhan.dropna(subset=["urgensi_manual", "urgensi_aturan"])
cm = pd.crosstab(dua["urgensi_manual"], dua["urgensi_aturan"]).reindex(index=URUT_URG, columns=URUT_URG, fill_value=0)
n = cm.values.sum()
po = np.trace(cm.values) / n
pe = (cm.sum(axis=1).values * cm.sum(axis=0).values).sum() / n ** 2
kappa = (po - pe) / (1 - pe) if pe < 1 else float("nan")
manual_tanpa_aturan = int(keluhan["urgensi_manual"].notna().sum() - n)

fig, ax = plt.subplots(figsize=(5, 4.2))
ax.imshow(cm.values, cmap=plt.matplotlib.colors.LinearSegmentedColormap.from_list("b", [NOL, SEQ[3]]))
for i in range(3):
    for j in range(3):
        v = cm.values[i, j]
        ax.text(j, i, str(v), ha="center", va="center", fontsize=11,
                color="#ffffff" if v > cm.values.max() * 0.5 else INK)
ax.set_xticks(range(3), URUT_URG)
ax.set_yticks(range(3), URUT_URG)
ax.set_xlabel("Label aturan")
ax.set_ylabel("Label manual")
ax.grid(False)
judul(ax, "Label aturan vs label manual", f"n = {n}, kesesuaian {po:.0%}, Cohen's kappa = {kappa:.2f}")
simpan(fig, "du_11_aturan_vs_manual")
plt.show()
print(f"Label manual tanpa label aturan (aturan tidak menemukan kata penanda): {manual_tanpa_aturan}")

In [ ]:
# 4.4 Kemungkinan laporan perbaikan petugas yang lolos sebagai "keluhan"
# Pola khas akun Dinas Bina Marga / Pemprov: tagar pasukan kuning, "TL" (tindak lanjut), dll.
POLA_PETUGAS = r"pasukankuning|welovejakarta|\btl\b|tindak lanjut|binamarga|bina marga"
pola_petugas = keluhan["text_clean"].fillna("").str.contains(POLA_PETUGAS)
print(f"Keluhan yang memuat pola posting petugas: {pola_petugas.sum()} dari {len(keluhan)} ({pola_petugas.mean():.0%})")
display(pd.crosstab(pola_petugas.rename("pola_petugas"), keluhan["urgensi"].fillna("Belum berlabel"))
        .reindex(columns=URUT_URG + ["Belum berlabel"], fill_value=0))
display(pd.crosstab(pola_petugas.rename("pola_petugas"), keluhan["tipe_akun"]))

In [ ]:
# 4.5 Ringkasan isu kualitas data (angka dihitung ulang setiap notebook dijalankan)
isu = pd.DataFrame([
    ("Lokasi", "Keluhan tanpa lokasi sama sekali", int((keluhan["level_lokasi"] == "tidak_ada").sum())),
    ("Lokasi", "Keluhan tanpa kelurahan pasti (tanpa kode kelurahan atau kelurahan tebakan)",
     int(keluhan["adm4_pcode"].isna().sum() + keluhan["ambigu_kelurahan"].astype(bool).sum())),
    ("Lokasi", "Kelurahan dengan 0 keluhan", int((agg_kel["jumlah_keluhan"] == 0).sum())),
    ("Lokasi", "Kecamatan dengan 0 keluhan", int((agg_kec["jumlah_keluhan"] == 0).sum())),
    ("Label", "Keluhan belum berlabel", int(keluhan["urgensi"].isna().sum())),
    ("Label", "Label manual", int(keluhan["urgensi_manual"].notna().sum())),
    ("Label", "Kelas Rendah (semua sumber label)", int((keluhan["urgensi"] == "Rendah").sum())),
    ("Label", "Kesesuaian aturan vs manual (%)", round(po * 100, 1)),
    ("Sumber", "Keluhan dari media/akun resmi", int((keluhan["tipe_akun"] == "media_resmi").sum())),
    ("Sumber", "Keluhan berpola posting petugas (kemungkinan laporan perbaikan)", int(pola_petugas.sum())),
    ("Waktu", "Tahun dengan keluhan paling sedikit", int(keluhan["tahun"].value_counts().idxmin())),
], columns=["aspek", "indikator", "nilai"], dtype=object)
isu.to_csv(TAB / "du_isu_kualitas.csv", index=False)
display(isu)

## 5. Ringkasan Temuan

> Angka di bawah sesuai data per **15 Oktober 2026** (hasil pipeline terakhir). Kalau pipeline dijalankan ulang, cek lagi angka di tabel/grafik di atas.

**Data yang tersedia**
- Dari 3.935 tweet mentah tersisa **807 keluhan jalan rusak** (2019 – Okt 2026) setelah deduplikasi, filter wilayah DKI, filter subjek kerusakan = jalan, dan pemisahan laporan perbaikan instansi (721 baris, disimpan terpisah sebagai layer pembanding).
- Tipe pengirim: **448 dari media/akun resmi**, 359 dari warga.

**Pola**
- Jumlah keluhan per tahun berfluktuasi; 2024 dan 2025 jauh lebih sedikit (36 dan 46), sedangkan 2026 paling banyak (177) karena scraping ulang per kecamatan.
- Jakarta Selatan dan Jakarta Pusat paling tinggi bila dinormalisasi per 100 ribu penduduk.

**Kualitas data yang memengaruhi modeling**
1. **Lokasi:** 37% keluhan (295) tidak punya lokasi dan hanya 76 dari 261 kelurahan yang punya keluhan. Di level kecamatan, 40 dari 42 kecamatan terisi. Analisis spasial lebih stabil di level **kecamatan**; level kelurahan dipakai untuk peta deskriptif.
2. **Label belum lengkap dan tidak seimbang:** 231 keluhan belum berlabel; kelas Rendah hanya 45.
3. **Label aturan dan label manual 100% sama** pada 159 keluhan yang punya keduanya. Ini menandakan aturan kata penanda disusun dari label manual tersebut, sehingga label manual yang ada **tidak bisa dipakai untuk menilai** label aturan maupun model yang dilatih dari label aturan. Untuk evaluasi yang jujur dibutuhkan sampel uji yang dilabeli manual tanpa melihat hasil aturan.
4. **Kata pembeda tiap kelas urgensi sama dengan kata penanda di aturan** (mis. *ambles*, *celaka* untuk Tinggi; *hati-hati*, *bahaya*, *segera* untuk Sedang). Model teks yang dilatih dari label aturan kemungkinan besar hanya akan meniru aturan tersebut.
5. **±11% keluhan (92) memuat pola posting petugas** (tagar pasukan kuning, "TL"/tindak lanjut, Bina Marga), hampir semuanya dari akun media/resmi. Sebagian kemungkinan laporan perbaikan yang lolos filter dan perlu dicek ulang sebelum modeling.
6. Mayoritas keluhan (448) dari akun media/resmi, sehingga pola kata bisa lebih mencerminkan gaya penulisan berita daripada keluhan warga. `tipe_akun` perlu dipertimbangkan sebagai fitur atau dipakai untuk analisis terpisah.
7. Rasio per 100 ribu penduduk di level **kelurahan tidak stabil**: kelurahan berpenduduk kecil (mis. Gambir, ±3 ribu jiwa) langsung melonjak hanya karena beberapa keluhan. Rasio lebih layak dipakai di level kecamatan/kota.